# 🎬 Longform Video Factory

Automated faceless whiteboard animation YouTube video pipeline.

**Workflow:**
1. Setup & credentials
2. Select topic from Google Sheet
3. Run Deep Research (Gemini)
4. Generate script (Claude/Gemini)
5. ⏸️ **Review script**
6. Generate voiceover (Fish Audio / Qwen3-TTS)
7. Generate scene illustrations (Gemini Flash Image)
8. Assemble video (FFmpeg)
9. Generate thumbnails
10. Generate SEO metadata
11. ⏸️ **Review video & metadata**
12. Finalize & Save status to Google Sheets


## Cell 0: Setup & Install


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone or update the repo
import os
REPO_DIR = '/content/longform'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/techafreshh/longform.git {REPO_DIR}

# Install dependencies
!pip install -q google-genai fish-audio-sdk openai gspread google-auth \
    python-dotenv requests Pillow openai-whisper youtube-transcript-api

# Add repo to path
import sys
sys.path.insert(0, REPO_DIR)

print('✅ Setup complete!')


Mounted at /content/drive
Cloning into '/content/longform'...
remote: Enumerating objects: 171, done.
remote: Counting objects: 100% (171/171), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 171 (delta 113), reused 125 (delta 67), pack-reused 0 (from 0)
Receiving objects: 100% (171/171), 81.71 KiB | 1.82 MiB/s, done.
Resolving deltas: 100% (113/113), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 4.0 MB/s eta 0:00:00
✅ Setup complete!


## Cell 1: Credentials & API Keys

API keys are automatically loaded from Google Colab's Secrets manager (`google.colab.userdata`). Alternatively, you can uncomment Option B and set them manually.


In [ ]:
# Configure API keys
import os

# Option A: Import secrets automatically using google.colab.userdata
try:
    from google.colab import userdata
    for key in ['GOOGLE_API_KEY', 'OPENROUTER_API_KEY', 'FISH_API_KEY', 'FISH_VOICE_ID', 'PEXELS_API_KEY', 'GOOGLE_SHEET_ID', 'USE_VERTEX', 'GCP_PROJECT', 'GCP_LOCATION']:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = str(val)
        except Exception:
            pass
    print('✅ Attempted importing secrets from Colab Userdata.')
except ImportError:
    print('ℹ️ Colab userdata not available. Using manually set environment variables.')

# Option B: Set directly (uncomment and fill if not using Colab Secrets)
# os.environ['GOOGLE_API_KEY'] = 'AIzaSy...'
# os.environ['OPENROUTER_API_KEY'] = 'sk-or-v1...'
# os.environ['FISH_API_KEY'] = '...'
# os.environ['FISH_VOICE_ID'] = '...'
# os.environ['PEXELS_API_KEY'] = '...'
# os.environ['GOOGLE_SHEET_ID'] = '...'
# Vertex AI options (uncomment to override Colab Secrets)
# os.environ['USE_VERTEX'] = 'true'
# os.environ['GCP_PROJECT'] = 'your-gcp-project-id'
# os.environ['GCP_LOCATION'] = 'us-central1'

# Authenticate with Google (for Sheets access)
try:
    from google.colab import auth
    auth.authenticate_user()
    print('✅ Authenticated with Google Account.')
except ImportError:
    print('ℹ️ Colab auth not available (running locally).')


✅ Attempted importing secrets from Colab Userdata.
✅ Authenticated with Google Account.


## Cell 2: Fetch Topics

Fetches all topics marked as `ready` in your Google Sheet.


In [ ]:
from src.sheets import get_ready_topics, update_status

# Fetch ready topics from your Google Sheet
try:
    topics = get_ready_topics()
    if not topics:
        print('⚠️  No topics with status "ready" found in the sheet.')
        print('   Add topics to your sheet or set one manually below.')
    else:
        print('\nSelect a topic by number:')
        for i, t in enumerate(topics):
            print(f'  [{i}] {t["topic"]} ({t["niche"]}, {t["style"]})')
except Exception as e:
    print(f'⚠️  Failed to load topics from Google Sheet: {e}')
    print('   You can define a topic manually in the next cell.')


## Cell 3: Pick Topic

Configure which topic to run (using the index from above or manual entry).


In [ ]:
#=== Pick your topic ===#
# Option A: From sheet (set the index from above)
TOPIC_INDEX = 0  # Change this to select a different topic

try:
    selected = topics[TOPIC_INDEX]
    TOPIC = selected['topic']
    NICHE = selected['niche']
    STYLE = selected['style']
    ADDITIONAL_PROMPT = selected['additional_prompt']
    TARGET_LENGTH = selected['target_length']
    SHEET_ROW = selected['row_number']
    print(f'✅ Selected from Sheet: "{TOPIC}"')
except (NameError, IndexError):
    # Option B: Manual entry
    TOPIC = "Why You Can\'t Remember Being a Baby"
    NICHE = "Psychology"
    STYLE = "color_whiteboard"  # or "chalkboard"
    ADDITIONAL_PROMPT = "Focus on hippocampus development"
    TARGET_LENGTH = "10-12 min"
    SHEET_ROW = None
    print(f'✅ Manual topic: "{TOPIC}"')

BASE_DIR = '/content/drive/MyDrive/LongformFactory'

# Option C: Shared Google Drive Folder Link (to continue/assemble on a different account)
# Paste Google Drive folder URL/ID here. If provided, existing scenes/voiceover will be downloaded automatically.
PROJECT_FOLDER_URL = "https://drive.google.com/drive/folders/1jKRKdYDUGL-nCYh8KvhStfY-4w2HReGY?usp=sharing"

# Option D: Resume from a specific scene index (optional)
# Set to an integer (e.g., 43) to skip generating/rendering scenes prior to that index.
RESUME_FROM_SCENE = None

if PROJECT_FOLDER_URL:
    print(f'   Shared Project URL configured: {PROJECT_FOLDER_URL}')
else:
    print(f'   Output folder will be: {BASE_DIR}/{NICHE}/{TOPIC[:40]}...')


✅ Manual topic: "Why You Can't Remember Being a Baby"
   Shared Project URL configured: https://drive.google.com/drive/folders/1jKRKdYDUGL-nCYh8KvhStfY-4w2HReGY?usp=sharing


## Cell 4: Initialize Paths

Sets up directory structures for the project.


In [ ]:
from pathlib import Path
from src.config import ProjectPaths, slugify
from src.gdrive import extract_gdrive_folder_id, get_gdrive_service, download_drive_folder_api, sync_gdrive_root_files, download_gdrive_subfolder_incremental

# Check if using a shared Google Drive folder link
gdrive_folder_id = extract_gdrive_folder_id(PROJECT_FOLDER_URL) if 'PROJECT_FOLDER_URL' in globals() and PROJECT_FOLDER_URL else None

if gdrive_folder_id:
    print(f'🔄 Shared Google Drive folder detected (ID: {gdrive_folder_id}).')

    # Authenticate and query the folder name from Google Drive API
    service = get_gdrive_service()
    gdrive_folder_name = None
    if service:
        try:
            folder_meta = service.files().get(fileId=gdrive_folder_id, fields="name").execute()
            gdrive_folder_name = folder_meta.get("name")
            print(f"📁 Retrieved remote Google Drive folder name: '{gdrive_folder_name}'")
        except Exception as e:
            print(f"⚠️ Could not fetch folder name from Drive API: {e}")

    if not gdrive_folder_name:
        gdrive_folder_name = slugify(TOPIC)

    # Target mounted Google Drive if available, otherwise fallback to volatile local path
    if Path('/content/drive/MyDrive').exists():
        local_project_dir = Path(f'/content/drive/MyDrive/LongformFactory/shared_projects/{gdrive_folder_name}')
    else:
        local_project_dir = Path(f'/content/shared_projects/{gdrive_folder_name}')

    if service:
        print('  Syncing project root files from Google Drive...')
        sync_gdrive_root_files(service, gdrive_folder_id, local_project_dir)

    # Override paths to point to this folder
    paths = ProjectPaths(
        base_dir=local_project_dir.parent,
        niche="",
        topic_slug=local_project_dir.name,
    )

    # Try loading metadata from scenes.json to override manual TOPIC/NICHE fallback
    scenes_json_path = local_project_dir / "scenes.json"
    if scenes_json_path.exists():
        import json
        try:
            with open(scenes_json_path, "r", encoding="utf-8") as f:
                scenes_info = json.load(f)
            TOPIC = scenes_info.get("topic", TOPIC)
            NICHE = scenes_info.get("niche", NICHE)
            STYLE = scenes_info.get("style", STYLE)

            # Reconstruct script_obj and scenes_data for downstream cells
            from src.scriptwriter import Script, Scene
            scenes_list = [
                Scene(
                    index=s["index"],
                    description=s["description"],
                    narration=s["narration"],
                    timestamp_hint=s.get("timestamp_hint")
                )
                for s in scenes_info.get("scenes", [])
            ]
            script_text = ""
            script_file_path = local_project_dir / "script.md"
            if script_file_path.exists():
                script_text = script_file_path.read_text(encoding="utf-8")

            script_obj = Script(
                raw_text=script_text,
                scenes=scenes_list,
                topic=TOPIC,
                niche=NICHE,
                style=STYLE
            )
            scenes_data = [s.to_dict() for s in script_obj.scenes]
            print(f"🔄 Automatically aligned notebook variables with shared project metadata:")
            print(f"   Topic: \"{TOPIC}\"")
            print(f"   Niche: \"{NICHE}\"")
            print(f"   Style: \"{STYLE}\"")
        except Exception as meta_err:
            print(f"⚠️ Failed to load scenes.json metadata: {meta_err}")
else:
    # Set up project path structure normally
    paths = ProjectPaths(
        base_dir=Path(BASE_DIR),
        niche=slugify(NICHE),
        topic_slug=slugify(TOPIC),
    )

paths.ensure_dirs()
print(f'✅ Project paths initialized at: {paths.project_dir}')


🔄 Shared Google Drive folder detected (ID: 1jKRKdYDUGL-nCYh8KvhStfY-4w2HReGY).
📁 Retrieved remote Google Drive folder name: 'the-strategic-genius-of-hannibal-how-he-outmaneuvered-rome-at-the-trebbia-river'
  Syncing project root files from Google Drive...
🔄 Automatically aligned notebook variables with shared project metadata:
   Topic: "The Strategic Genius of Hannibal: How He Outmaneuvered Rome at the Trebbia River"
   Niche: "Ancient History"
   Style: "color_whiteboard"
✅ Project paths initialized at: /content/drive/MyDrive/LongformFactory/shared_projects/the-strategic-genius-of-hannibal-how-he-outmaneuvered-rome-at-the-trebbia-river


## Cell 5: Stage 1 - Deep Research

Uses Gemini Deep Research to write a research document. If the research file already exists, it is loaded from disk to save credits.


In [ ]:
from src.pipeline import run_stage_research

# Force regeneration: set to True if you want to overwrite existing research
FORCE_RESEARCH = False

if SHEET_ROW:
    update_status(SHEET_ROW, 'researching')

research_text = run_stage_research(
    paths=paths,
    topic=TOPIC,
    niche=NICHE,
    additional_prompt=ADDITIONAL_PROMPT,
    force=FORCE_RESEARCH,
    gdrive_folder_id=gdrive_folder_id if 'gdrive_folder_id' in globals() else None,
)


ℹ️ Research file already exists at: /content/drive/MyDrive/LongformFactory/shared_projects/the-strategic-genius-of-hannibal-how-he-outmaneuvered-rome-at-the-trebbia-river/research.md
   Skipping research stage to save credits. Set force=True to regenerate.


## Cell 6: Stage 2 - Script Generation

Generates the narration script with embedded scene markers. You can change the model here (e.g. `anthropic/claude-3-5-sonnet`). If the script already exists, it is loaded from disk to save credits.


In [ ]:
from src.pipeline import run_stage_script, run_stage_reference_scripts
from src.gdrive import download_gdrive_subfolder_incremental

# Configure scriptwriting model. Default is Claude Sonnet via OpenRouter.
SCRIPT_MODEL = 'anthropic/claude-3-5-sonnet'
FORCE_SCRIPT = False
FORCE_REFERENCE_FETCH = False

# Sync reference scripts from shared Google Drive folder if active
if gdrive_folder_id and 'service' in globals() and service:
    print('🔄 Fetching reference scripts incrementally from Google Drive...')
    download_gdrive_subfolder_incremental(service, gdrive_folder_id, 'reference_scripts', paths.project_dir)

# Fetch 3 reference scripts on the topic from YouTube
run_stage_reference_scripts(
    paths=paths,
    topic=TOPIC,
    count=3,
    force=FORCE_REFERENCE_FETCH,
    gdrive_folder_id=gdrive_folder_id if 'gdrive_folder_id' in globals() else None,
)

script_obj = run_stage_script(
    paths=paths,
    topic=TOPIC,
    niche=NICHE,
    research_text=research_text,
    style=STYLE,
    target_length=TARGET_LENGTH,
    additional_prompt=ADDITIONAL_PROMPT,
    model=SCRIPT_MODEL,
    force=FORCE_SCRIPT,
    gdrive_folder_id=gdrive_folder_id if 'gdrive_folder_id' in globals() else None,
)

if SHEET_ROW:
    update_status(SHEET_ROW, 'review_script')

print(f'✅ Script generation complete!')
print(f'   Scene count: {script_obj.scene_count}')


ℹ️ Script and scenes list already exist at: /content/drive/MyDrive/LongformFactory/shared_projects/the-strategic-genius-of-hannibal-how-he-outmaneuvered-rome-at-the-trebbia-river/script.md
   Skipping scriptwriting stage to save credits. Set force=True to regenerate.
✅ Script generation complete!
   Scene count: 86


## Cell 7: ⏸️ Script Review Gate

Review the generated script below. If you want to make manual changes, open the script file on Drive, edit it, and save it. If you want to regenerate it entirely, change settings in Cell 6 and re-run Cell 6.


In [ ]:
from IPython.display import Markdown, display

# Display the script for manual review
if paths.script_file.exists():
    script_text = paths.script_file.read_text(encoding='utf-8')
    display(Markdown(script_text))
else:
    print('⚠️ Script file not found')

print('\n' + '='*60)
print('👆 Review the script above.')
print('   If you want to edit it, open the file on Google Drive and save it.')
print('   If you want to regenerate it, change SCRIPT_MODEL/prompts in Cell 6 and re-run.')


## Cell 8: Stage 3 - Voice Generation

Generates the voiceover track. If the voiceover already exists, it is loaded from disk to save credits.


In [ ]:
from src.pipeline import run_stage_voice

# TTS Engine settings
TTS_ENGINE = 'fish'           # 'fish', 'qwen_1.7b', or 'qwen_0.6b'
VOICE_MODEL = 's2.1-pro-free' # 's2.1-pro-free' or other Fish model version
REFERENCE_AUDIO = None        # Reference audio path for Qwen cloning
FORCE_VOICE = False

if SHEET_ROW:
    update_status(SHEET_ROW, 'generating')

scenes_data = [s.to_dict() for s in script_obj.scenes]

# Granular fetch-on-demand for voice stage
if gdrive_folder_id and 'service' in globals() and service:
    print('🔄 Fetching existing voice assets incrementally...')
    download_gdrive_subfolder_incremental(service, gdrive_folder_id, 'audio', paths.project_dir)

voice_result = run_stage_voice(
    paths=paths,
    scenes_data=scenes_data,
    tts_engine=TTS_ENGINE,
    voice_model=VOICE_MODEL,
    reference_audio=REFERENCE_AUDIO,
    force=FORCE_VOICE,
    gdrive_folder_id=gdrive_folder_id if 'gdrive_folder_id' in globals() else None,
)

print(f'✅ Voiceover duration: {voice_result.total_duration:.1f}s')


🔄 Fetching existing voice assets incrementally...
ℹ️ Voiceover audio already exists at: /content/drive/MyDrive/LongformFactory/shared_projects/the-strategic-genius-of-hannibal-how-he-outmaneuvered-rome-at-the-trebbia-river/audio/voiceover.wav
   Skipping voice generation stage to save credits. Set force=True to regenerate.
✅ Voiceover duration: 1222.3s


## Cell 9: Stage 4 - Scene Image Generation

Generates the whiteboard/chalkboard illustration for each scene. By default, existing images are skipped to save credits. You can list specific scenes in `REGENERATE_SCENES` to force regenerate only those.


In [ ]:
from src.pipeline import run_stage_scenes

FORCE_SCENES = False
REGENERATE_SCENES = None

# Granular fetch-on-demand for scene image stage
if gdrive_folder_id and 'service' in globals() and service:
    print('🔄 Fetching existing scene images incrementally...')
    download_gdrive_subfolder_incremental(service, gdrive_folder_id, 'scenes', paths.project_dir)
  # List of specific scene indices to regenerate, e.g. [1, 5, 12] or None

image_paths = run_stage_scenes(
    paths=paths,
    scenes_data=scenes_data,
    style=STYLE,
    force=FORCE_SCENES,
    force_scenes=REGENERATE_SCENES,
    resume_from_scene=RESUME_FROM_SCENE if 'RESUME_FROM_SCENE' in globals() else None,
    gdrive_folder_id=gdrive_folder_id if 'gdrive_folder_id' in globals() else None,
)

print(f'✅ Generated/Found {len(image_paths)} scene images.')


🔄 Fetching existing scene images incrementally...
ℹ️ All expected scene images already exist in: /content/drive/MyDrive/LongformFactory/shared_projects/the-strategic-genius-of-hannibal-how-he-outmaneuvered-rome-at-the-trebbia-river/scenes
   Skipping image generation to save credits. Set force=True to regenerate.
✅ Generated/Found 86 scene images.


## Cell 10: Stage 5 - Video Assembly

Assembles the final video using FFmpeg, applying Ken Burns and transitions, mixing voiceover and BGM, and burning subtitles. If the final video already exists, it is skipped.


In [ ]:
from src.pipeline import run_stage_assembly

BGM_PATH = None       # Path to background music (or None)
BGM_VOLUME = 0.15     # BGM volume (0-1)
KEN_BURNS = True      # Apply zoom/pan animation
FORCE_ASSEMBLY = False
RENDER_WORKERS = 4    # Number of parallel workers to render scene clips
SKIP_SUBTITLES = True  # Skip subtitle generation and burn-in by default

# Granular fetch-on-demand for video assembly stage
if gdrive_folder_id and 'service' in globals() and service:
    print('🔄 Fetching existing rendering clips and assets incrementally...')
    download_gdrive_subfolder_incremental(service, gdrive_folder_id, 'clips_cache', paths.project_dir.parent)
    download_gdrive_subfolder_incremental(service, gdrive_folder_id, 'audio', paths.project_dir)
    download_gdrive_subfolder_incremental(service, gdrive_folder_id, 'scenes', paths.project_dir)

final_video_path = run_stage_assembly(
    paths=paths,
    scenes_data=scenes_data,
    voice_result=voice_result,
    style=STYLE,
    bgm_path=BGM_PATH,
    bgm_volume=BGM_VOLUME,
    ken_burns=KEN_BURNS,
    force=FORCE_ASSEMBLY,
    resume_from_scene=RESUME_FROM_SCENE if 'RESUME_FROM_SCENE' in globals() else None,
    max_workers=RENDER_WORKERS,
    skip_subtitles=SKIP_SUBTITLES,
    gdrive_folder_id=gdrive_folder_id if 'gdrive_folder_id' in globals() else None,
)

print(f'✅ Video assembled at: {final_video_path}')


🔄 Fetching existing rendering clips and assets incrementally...
  📥 Downloading: clip_02_dur_13.82.mp4 (13.91 MB)...
  📥 Downloading: joined.mp4 (0.00 MB)...
  📥 Downloading: clip_86_dur_32.37.mp4 (2.60 MB)...
  📥 Downloading: clip_85_dur_20.64.mp4 (1.14 MB)...
  📥 Downloading: clip_84_dur_12.59.mp4 (1.58 MB)...
  📥 Downloading: clip_83_dur_12.09.mp4 (2.23 MB)...
  📥 Downloading: clip_82_dur_13.06.mp4 (1.36 MB)...
  📥 Downloading: clip_81_dur_16.64.mp4 (2.45 MB)...
  📥 Downloading: clip_80_dur_11.44.mp4 (3.72 MB)...
  📥 Downloading: clip_79_dur_13.24.mp4 (3.86 MB)...
  📥 Downloading: clip_78_dur_14.31.mp4 (8.57 MB)...
  📥 Downloading: clip_77_dur_18.13.mp4 (5.96 MB)...
  📥 Downloading: clip_76_dur_16.09.mp4 (4.82 MB)...
  📥 Downloading: clip_75_dur_13.82.mp4 (4.34 MB)...
  📥 Downloading: clip_74_dur_15.75.mp4 (7.97 MB)...
  📥 Downloading: clip_73_dur_14.24.mp4 (2.75 MB)...
  📥 Downloading: clip_72_dur_11.86.mp4 (5.63 MB)...
  📥 Downloading: clip_71_dur_14.84.mp4 (9.09 MB)...
  📥 Downlo

  Uploading new file 'the-strategic-genius-of-hannibal-how-he-outmaneuvered-rome-at-the-trebbia-river.mp4' to Google Drive...
  Updating existing file 'subtitles.srt' on Google Drive...
✅ Video assembled: /content/drive/MyDrive/LongformFactory/shared_projects/the-strategic-genius-of-hannibal-how-he-outmaneuvered-rome-at-the-trebbia-river/output/the-strategic-genius-of-hannibal-how-he-outmaneuvered-rome-at-the-trebbia-river.mp4
   Duration: 1179.9s | Size: 807.9 MB
✅ Video assembled at: /content/drive/MyDrive/LongformFactory/shared_projects/the-strategic-genius-of-hannibal-how-he-outmaneuvered-rome-at-the-trebbia-river/output/the-strategic-genius-of-hannibal-how-he-outmaneuvered-rome-at-the-trebbia-river.mp4


## Compress the video for human review


In [ ]:
import subprocess
import os
from pathlib import Path

# Define output path for the extreme compressed video
compressed_video_path = final_video_path.parent / f"ultra_compressed_{final_video_path.name}"

print(f"🚀 Starting Extreme Compression (Target: <100MB)...")

# Calculate bitrates for ~20 min video to hit <100MB
# Total budget approx 670kbps. Video 550k, Audio 48k.
command = [
    'ffmpeg', '-y',
    '-i', str(final_video_path),
    '-c:v', 'h264_nvenc',
    '-b:v', '550k',
    '-maxrate:v', '700k',
    '-bufsize:v', '1000k',
    '-preset', 'slow',
    '-c:a', 'aac',
    '-b:a', '48k',
    str(compressed_video_path)
]

try:
    # Added capture_output=True to see the logs if it fails
    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"❌ FFmpeg Error:\n{result.stderr}")
    elif not compressed_video_path.exists():
        print("❌ Process finished but output file was not created.")
    else:
        new_size = compressed_video_path.stat().st_size / (1024*1024)
        print(f"✅ Compression complete!")
        print(f"Original Size: {final_video_path.stat().st_size / (1024*1024):.2f} MB")
        print(f"New Size: {new_size:.2f} MB")

        if new_size > 100:
            print("⚠️ Still over 100MB. Lowering bitrate further or resolution is recommended.")
        else:
            print("🎯 Goal achieved: Under 100MB!")

except Exception as e:
    print(f"⚠️ Script Error: {e}")

🚀 Starting Extreme Compression (Target: <100MB)...
✅ Compression complete!
Original Size: 807.93 MB
New Size: 101.27 MB
⚠️ Still over 100MB. Lowering bitrate further or resolution is recommended.


## Cell 11: Stage 6 - Thumbnail Generation

Generates 3 YouTube thumbnail options. Skipped if they already exist.


In [ ]:
from src.pipeline import run_stage_thumbnails

FORCE_THUMBNAILS = False

# Granular fetch-on-demand for thumbnail stage
if gdrive_folder_id and 'service' in globals() and service:
    print('🔄 Fetching existing thumbnails incrementally...')
    download_gdrive_subfolder_incremental(service, gdrive_folder_id, 'thumbnails', paths.project_dir)

thumbnail_paths = run_stage_thumbnails(
    paths=paths,
    topic=TOPIC,
    niche=NICHE,
    style=STYLE,
    force=FORCE_THUMBNAILS,
    gdrive_folder_id=gdrive_folder_id if 'gdrive_folder_id' in globals() else None,
)

print(f'✅ Generated/Found {len(thumbnail_paths)} thumbnail variants.')


## Cell 12: Stage 7 - SEO Metadata

Generates YouTube title, tags, description, and category. Skipped if it already exists.


In [ ]:
from src.pipeline import run_stage_seo

FORCE_SEO = False

seo_data = run_stage_seo(
    paths=paths,
    topic=TOPIC,
    niche=NICHE,
    script_text=script_obj.raw_text,
    style=STYLE,
    force=FORCE_SEO,
    gdrive_folder_id=gdrive_folder_id if 'gdrive_folder_id' in globals() else None,
)

print(f'✅ SEO Metadata generated!')


## Cell 13: ⏸️ Video & Thumbnail Review Gate

Watch the preview of the video, thumbnails, and SEO text before publishing.


In [ ]:
from IPython.display import Video, display, Image
from pathlib import Path

# Preview the generated video
if final_video_path.exists():
    print(f'🎬 Video path: {final_video_path}')
    print(f'   Duration: {voice_result.total_duration:.1f}s')

    try:
        display(Video(str(final_video_path), embed=True, width=800))
    except Exception:
        print('   (Preview player not available - check file on Google Drive)')
else:
    print('⚠️ Video file not found')

# Preview the thumbnails
if thumbnail_paths:
    print('\n🖼️ Thumbnail variants:')
    for tp in thumbnail_paths:
        if tp.exists():
            display(Image(filename=str(tp), width=400))

# Preview the SEO metadata
print('\n📊 SEO Metadata:')
print(f'   Title: {seo_data.get("title", "N/A")}')
print(f'   Description: {seo_data.get("description", "N/A")[:200]}...')
print(f'   Tags: {json.dumps(seo_data.get("tags", []))}')


## Cell 14: Finalize & Mark Complete

Mark the video as complete in the Google Sheet and upload path metadata.


In [ ]:
from src.sheets import update_status
from src.gdrive import get_gdrive_service, sync_local_outputs_to_drive

# Sync outputs back to the shared Google Drive folder if configured
if 'gdrive_folder_id' in globals() and gdrive_folder_id:
    service = get_gdrive_service()
    if service:
        sync_local_outputs_to_drive(service, gdrive_folder_id, paths)

if SHEET_ROW:
    drive_link = str(final_video_path)
    thumbnail_link = str(thumbnail_paths[0]) if thumbnail_paths else ""

    update_status(
        SHEET_ROW,
        'complete',
        extra_data={
            'drive_link': drive_link,
            'thumbnail_link': thumbnail_link,
            'seo_title': seo_data.get('title', ''),
            'seo_description': seo_data.get('description', ''),
            'seo_tags': ', '.join(seo_data.get('tags', [])) if isinstance(seo_data.get('tags'), list) else seo_data.get('tags', '')
        }
    )
    print("✅ Spreadsheet updated and topic marked 'complete'!")
else:
    print("ℹ️ Run was started manually. Output saved to Drive but Sheet not updated.")


## 🛠️ Utilities


In [ ]:
# === UTILITY: Clone your voice on Fish Audio ===
# Run this ONCE with your voice sample, then save the voice_id

# from src.voice import clone_voice_fish
# voice_id = clone_voice_fish(
#     audio_path='/content/drive/MyDrive/my_voice_sample.wav',
#     voice_name='my-narrator-voice',
# )
# print(f'Set this in your config: FISH_VOICE_ID={voice_id}')


In [ ]:
# === UTILITY: Initialize Google Sheet template ===
# Run this ONCE to set up your content calendar headers

# from src.sheets import create_sheet_template
# create_sheet_template()


### Utility: List available Imagen models

Check which Imagen models are currently accessible via your configured API key or Google Cloud Vertex AI project.

In [ ]:
# === UTILITY: Diagnostic & Enablement for Vertex AI & Google AI Studio ===
import os
from google import genai
from src.config import GOOGLE_API_KEY, GCP_PROJECT, GCP_LOCATION

print("=== Checking Google AI Studio (API Key) ===")
api_key = GOOGLE_API_KEY or os.getenv("GOOGLE_API_KEY")
if api_key:
    try:
        client = genai.Client(api_key=api_key)
        models = client.models.list()
        found = False
        for m in models:
            if "imagen" in m.name.lower():
                print(f"  - {m.name}")
                found = True
        if not found:
            print("  No Imagen models returned by AI Studio.")
    except Exception as e:
        print(f"  ❌ Error querying AI Studio: {e}")
else:
    print("  (GOOGLE_API_KEY not configured)")

print("\n=== Checking Vertex AI ===")
project = GCP_PROJECT or os.getenv("GCP_PROJECT")
location = GCP_LOCATION or os.getenv("GCP_LOCATION", "us-central1")
if project:
    print(f"Project ID: {project}")
    print(f"Location: {location}")

    # Try to enable Vertex AI API via gcloud
    print("\nAttempting to ensure Vertex AI API is enabled...")
    try:
        import subprocess
        res = subprocess.run(
            ["gcloud", "services", "enable", "aiplatform.googleapis.com", f"--project={project}"],
            capture_output=True, text=True
        )
        if res.returncode == 0:
            print("  ✅ Vertex AI API is enabled (or has been enabled successfully)!")
        else:
            print(f"  ⚠️ gcloud could not enable Vertex AI API. Error:\n{res.stderr}")
            print("  Please ensure your active Colab user has 'Owner' or 'Service Usage Admin' roles on the project.")
    except Exception as ge:
        print(f"  Could not run gcloud service check: {ge}")

    try:
        client = genai.Client(vertexai=True, project=project, location=location)
        models = client.models.list()
        found = False
        for m in models:
            if "imagen" in m.name.lower():
                print(f"  - {m.name}")
                found = True
        if not found:
            print("  No Imagen models returned by Vertex AI. (This is normal if listing is restricted, but direct generation may still work!)")
    except Exception as e:
        print(f"  ❌ Error querying Vertex AI: {e}")
        print("\n💡 Troubleshooting Tips for Vertex AI:")
        print("  1. Make sure you authenticated with the Google Account that has access to the GCP project.")
        print("  2. Verify that billing is active on your GCP project (required for Vertex AI, even with welcome credits).")
        print("  3. Go to the GCP Console (https://console.cloud.google.com/vertex-ai/publishers/google/models/imagen-3.0-generate-002) and check if you need to agree to terms or enable access.")
else:
    print("  (Vertex AI project details not configured)")